In [3]:
from langchain_core.tools import tool

@tool
async def division(a: int, b: int) -> float:
    """Divides two numbers."""
    return a / b

In [4]:
print(division.name)  # Output: division
print(division.description)  # Output: Divides two numbers.         

division
Divides two numbers.


In [5]:
from typing import Annotated,List

@tool
def multiply_by_max(
    a: Annotated[int, "First number"],
    b: Annotated[List[int], "Second number"],
) -> int:
    """Multiplies two numbers and returns the result."""
    return a * max(b)


In [6]:
print(multiply_by_max.args)
multiply_by_max.invoke({"a": 5, "b": [1, 2, 3]})

{'a': {'description': 'First number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


15

In [7]:
from langchain_core.tools import StructuredTool

def multiply (a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

async def amultiply (a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

calculator_tool = StructuredTool.from_function(func=multiply, coroutine=amultiply)

print(calculator_tool.invoke({"a": 5, "b": 3}))
print(await calculator_tool.ainvoke({"a": 5, "b": 9}))

15
45


In [8]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query": "What is Anti-Matter?"}))



Page: Antimatter
Summary: In modern physics, antimatter is defined as matter composed of the antiparticles (or "partners") of the corresponding particles in "ordinary" matter, and can be thought of as matter with reversed charges and parity, or going backward in time (see CPT symmetry). Antimatter occurs in natural processes like cosmic ray collisions and some types of radioactive decay, but only a tiny fraction of these have successfully been bound together in experiments to form antiatoms. Min


In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["AZURE_OPENAI_ENDPOINT"]
os.environ["AZURE_OPENAI_API_KEY"]
os.environ["TAVILY_API_KEY"]

from langchain_tavily import TavilySearch

tool_search = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # start_date=None,
    # end_date=None,
    # include_domains=None,
    # exclude_domains=None,
    # include_usage= False
)

print(tool_search.invoke({"query": "What is the latest news on AI?"}))

{'query': 'What is the latest news on AI?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.reuters.com/technology/artificial-intelligence/', 'title': 'AI News | Latest Headlines and Developments', 'content': 'Explore the latest artificial intelligence news with Reuters - from AI breakthroughs and technology trends to regulation, ethics, business and global', 'score': 0.767638, 'raw_content': None}, {'url': 'https://www.wsj.com/tech/ai', 'title': 'Artificial Intelligence - Latest AI News and Analysis', 'content': 'Artificial Intelligence · Anthropic and FIS Are Building an AI Agent to Help Banks Police Financial Crimes · White House Officials Discuss Assessing AI Models', 'score': 0.7409441, 'raw_content': None}, {'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates', 'title': 'Agentic AI News + AI Breakthroughs + AI Developments', 'content': 'Critically, responsible AI is not keeping pace: documented AI incidents rose to 362, mod

In [10]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Adds two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a *  b


In [11]:
tools = [add , multiply ,wiki_tool, tool_search]
tools

[StructuredTool(name='add', description='Adds two numbers.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x00000279A72C1EE0>),
 StructuredTool(name='multiply', description='Multiplies two numbers.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x00000279A6E37920>),
 WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\KrishnaShukla\\Desktop\\Gen-AI\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=5, lang='en', load_all_available_meta=False, doc_content_chars_max=500)),
 TavilySearch(max_results=5, topic='general', api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()
from langchain_openai import AzureChatOpenAI
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    deployment_name="gpt-4o",
    api_version="2024-02-15-preview"
)

llm_tools_binded = llm.bind_tools(tools)

query="What is the latest news on AI and what is the product of 5 and 3?"
response = llm_tools_binded.invoke(query)
print(response)



content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 1308, 'total_tokens': 1366, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 12, 'engine_ttft_ms': 145, 'engine_ttlt_ms': 840, 'pre_inference_ms': 193, 'service_tbt_ms': 12, 'service_ttft_ms': 913, 'service_ttlt_ms': 1626, 'total_duration_ms': 1430, 'user_visible_ttft_ms': 721}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-Dc6EZma2n5MDVrrxg5SoDukQL7O4U', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 's

In [16]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 1308, 'total_tokens': 1366, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 12, 'engine_ttft_ms': 145, 'engine_ttlt_ms': 840, 'pre_inference_ms': 193, 'service_tbt_ms': 12, 'service_ttft_ms': 913, 'service_ttlt_ms': 1626, 'total_duration_ms': 1430, 'user_visible_ttft_ms': 721}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-Dc6EZma2n5MDVrrxg5SoDukQL7O4U', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 's

In [14]:
for tool_call in response.tool_calls:
  print(tool_call["name"])
  print(tool_call["args"])



tavily_search
{'query': 'latest news on AI', 'time_range': 'day', 'topic': 'news'}
multiply
{'a': 5, 'b': 3}
